In [1]:
import bilby
from bilby.core.utils import random
from pprint import pprint as pp
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import bilby.gw.conversion as conv
from bilby.gw.conversion import polytrope_or_causal_params_to_lambda_1_lambda_2


from gwbench import Network, injections_CBC_params_redshift, M_of_Mc_eta, f_isco_Msolar
import corner

from converse_likelihood import EOSHyperparameterLikelihood

import lalsimulation as lalsim

from scipy.stats import gaussian_kde

/Users/ved/miniforge3/envs/neutron/lib/python3.12/site-packages/gwbench/basic_relations.py:20: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  from lal import GreenwichMeanSiderealTime


In [3]:
Msun = 1.989e30

def is_viable(g0, g1, g2, lp1=35.293, lp2=35.649, min_max_mass=2.0):
    p1_si, p2_si = lp1 - 1., lp2 - 1.
    if lalsim.SimNeutronStarEOS3PDViableFamilyCheck(g0, p1_si, g1, p2_si, g2, 0) != 0:
        return False
    eos = lalsim.SimNeutronStarEOS3PieceDynamicPolytrope(g0, p1_si, g1, p2_si, g2)
    max_h = lalsim.SimNeutronStarEOSMaxPseudoEnthalpy(eos)
    cs = lalsim.SimNeutronStarEOSSpeedOfSoundGeometerized(max_h, eos)
    if cs > 1.1:
        return False
    family = lalsim.CreateSimNeutronStarFamily(eos)
    max_mass = lalsim.SimNeutronStarMaximumMass(family) / Msun
    if max_mass < min_max_mass:
        return False
    return True

# rejection sample
n_target = 100
viable = []
n_total = 0
while len(viable) < n_target:
    g0, g1, g2 = np.random.uniform(1.0, 5.0, 3)
    n_total += 1
    if is_viable(g0, g1, g2):
        viable.append((g0, g1, g2))

viable = np.array(viable)
print(f"Acceptance rate: {len(viable)/n_total:.3%}")


Acceptance rate: 17.575%


In [ ]:
np.save('viable_gamma_samples.npy', viable)

kde = gaussian_kde(viable.T, bw_method='scott')

In [ ]:
# def diagnose_eos(param1, log10_p1_cgs, param2, log10_p2_cgs, param3, m, causal=0):
#     import lalsimulation as lalsim
#     import numpy as np
    
#     G = 6.674e-11
#     c = 3e8
#     Msun = 1.989e30

#     p1_si = log10_p1_cgs - 1.
#     p2_si = log10_p2_cgs - 1.

#     if log10_p1_cgs >= log10_p2_cgs:
#         print("FAIL: pressure ordering violated")
#         return

#     viable = lalsim.SimNeutronStarEOS3PDViableFamilyCheck(
#         param1, p1_si, param2, p2_si, param3, causal)
#     print(f"ViableFamilyCheck: {viable}  (0 = pass)")

#     eos = lalsim.SimNeutronStarEOS3PieceDynamicPolytrope(
#         param1, p1_si, param2, p2_si, param3)
#     family = lalsim.CreateSimNeutronStarFamily(eos)

#     max_h = lalsim.SimNeutronStarEOSMaxPseudoEnthalpy(eos)
#     cs    = lalsim.SimNeutronStarEOSSpeedOfSoundGeometerized(max_h, eos)
#     print(f"Max sound speed / c: {cs:.4f}  (must be <= 1.1)")

#     min_mass = lalsim.SimNeutronStarFamMinimumMass(family) / Msun
#     max_mass = lalsim.SimNeutronStarMaximumMass(family) / Msun
#     print(f"Mass range: [{min_mass:.3f}, {max_mass:.3f}] Msun")
#     print(f"Requested mass: {m:.3f} Msun  in range: {min_mass <= m <= max_mass}")

# diagnose_eos(param1, log10_p1_cgs, param2, log10_p2_cgs, param3, m=1.4)